# NL experiment · Evaluation 2: every model per country, and every input against Landsat in each country

The second of the three evaluation notebooks. It reads the test predictions that 16 collected (`eval_runs.csv`, `eval_test_predictions.csv.gz` in `COLLECTED_DIR`) and evaluates every run **per country**. Then, in every country, it compares every input with the **baseline, Landsat** (12; the row `base` of 04/05; the row `landsat` of 07/08), always with the same model and start, so the runs from pretrained weights are compared with their pretrained baseline and the runs from scratch with their scratch baseline.

**The target** is that of 16: the average nighttime light of the patch (`nl` of 02: clipped to 0–150 and normalised with `NormFunc` of the survey year), predicted from Landsat (and the MODIS NDVI bands of each notebook) only; RMSE, MAE and bias are in target units. DMSP (surveys up to 2013) and VIIRS (from 2014) targets are on different scales, so in a country with surveys of both periods the R² also counts how well a model tells the periods apart. `NL_PERIOD` (`EVAL_NL_PERIOD=DMSP` or `VIIRS` in the environment) evaluates the test clusters of one period only, with every table and plot in `COLLECTED_DIR/<period>_clusters/` (`EVAL_DIR` points there); runs with fewer than `MIN_N` clusters of that period (04 and 05 have only VIIRS clusters) are left out.

**Per country**, on the pooled test predictions of the five folds (03 spreads every country over all folds, so every cluster is predicted by the model that did not see it):

* `R2`: 1 − SSE / SST against the country's **own** mean target, so it measures how well the model ranks the clusters within the country; negative when the predictions miss the country's level or spread. `r2`: the squared correlation of predicted and observed target (Yeh et al. 2020), which an offset does not lower. `RMSE`, `MAE`, `bias` (mean of predicted − observed) in target units. R² needs at least `MIN_N` test clusters.
* **Against the baseline** (section 3): `dR2` = R²(input) − R²(Landsat) on the clusters of the country that both runs have (paired), with its 95 % interval from `BOOTSTRAP` Poisson resamples of 03's spatial groups, `p`, `q` (Benjamini–Hochberg over the countries of one model, input and start), `dRMSE` and `dMAE` (negative: the input helps), and `folds better` (test folds with at least `MIN_FOLD_N` of the country's clusters in which the input has the higher R²). `verdict`: `better` or `worse` when the interval excludes 0.
* **Over the models** (`country_consensus.csv`): for every start, family, input and country, the median ΔR² of the models (without `SUMMARY_SKIP`: `satclip_only` reads no band, so its ΔR², `noise dR2`, shows the training noise; the baseline models VGG-16-BN and MLP of 11–15 are shown on their own, `vgg16bn dR2` and `mlp dR2`), how many models gain (`models better`), gain or lose with an interval that excludes 0 (`sig. better`, `sig. worse`), and the model with the largest gain.

**Sections:** 2. scores of every run per country; 3. every input against the baseline per country; 4. figures per country; 5. the best input per country; 6. a map of the gains; 7. every model against the **baseline models** VGG-16-BN and MLP (`BASELINE_MODELS`) per country, with the same input and against the baseline model on Landsat.

**Output** in `EVAL_DIR`: `country_metrics.csv`, `country_vs_baseline.csv`, `country_consensus.csv`, `country_best_input.csv`, `country_vs_baseline_models.csv`, `country_vs_baseline_models_summary.csv`, and the plots `country_r2_baseline.png`, `country_consensus_<start>.png`, `country_vs_baseline_<start>_<family>.png`, `country_map_<start>.png`, `country_vs_baseline_models_<start>_<baseline model>.png`, `country_vs_<baseline model>_<start>.png`.

**Running.** After 16, on a CPU (a few minutes).

# Import libraries

In [ ]:
import os
import re
import glob
import json
import time
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from IPython.display import display, Image

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
DATA_DIR  = '/path/to/data/'   # the IWI study: Dataset/, folds, caches
WORK_DIR  = os.path.join(DATA_DIR, 'WaveletNL')                  # this study: the run folders of 04-15
FOLDS_CSV = os.path.join(WORK_DIR, 'train_clusters_folds_nl.csv')   # written by 03_Folds_NL_Target
PERIODS   = 'DMSP_VIIRS'   # the run folders of 07, 08 and 11-15 that are read, <periods>_<start>...: both NL periods,
                           # as in 07, 08 and 11-15 (their clusters; the NL image itself is the target, not an input)
COLLECTED_DIR = os.path.join(WORK_DIR, 'evaluation', PERIODS)   # 16 writes the collected predictions here
NL_PERIOD = os.environ.get('EVAL_NL_PERIOD') or None   # None: all test clusters; 'DMSP' or 'VIIRS': only the clusters
                           # of that NL period, whose targets are on one scale (NormFunc of the period), with the
                           # tables and plots in COLLECTED_DIR/<NL_PERIOD>_clusters/ (EVAL_NL_PERIOD=VIIRS in the environment)
EVAL_DIR  = COLLECTED_DIR if NL_PERIOD is None else os.path.join(COLLECTED_DIR, f'{NL_PERIOD}_clusters')   # output

# ---- evaluation ----------------------------------------------------------------------------------
TARGET       = 'nl'            # the label of FOLDS_CSV that every model predicts: the NL target of 02, the mean of the
                               # patch's NL image of the survey year, clipped to (0, 150) and normalised with NormFunc
                               # of the survey year (DMSP up to 2013, VIIRS from 2014): RMSE, MAE and bias are in
                               # these units ('target units')
FOLDS        = ['A', 'B', 'C', 'D', 'E']
BASELINE     = 'L'             # the baseline input: the 7 Landsat bands (12; the row 'base' of 04 and 05; the row
                               # 'landsat' of 07 and 08)
BOOTSTRAP    = 1000            # resamples of 03's spatial groups for the 95 % intervals (0: no intervals)
SEED         = 0               # the resamples: the same for every comparison, so all comparisons are paired
MIN_N        = 30              # an R2 needs at least this many test clusters (a country, its rural or urban clusters)
MIN_FOLD_N   = 10              # a test fold counts in 'folds better' with at least this many of the clusters
NOISE_MODELS = ['satclip_only']   # read no band, only the coordinates: their change between inputs is training noise
                                  # (left out of the summaries over models, shown as 'noise dR2')
BASELINE_MODELS = ['vgg16bn', 'mlp']   # the baseline models of 11-15 (VGG-16-BN and an MLP on band statistics, on
                                       # the same inputs): in every table and plot of the models and compared with
                                       # every model (16, section 6), but left out of the summaries over models
                                       # (shown there on their own: 'vgg16bn dR2', 'mlp dR2'), so these stay those of
                                       # the ten models
SUMMARY_SKIP = NOISE_MODELS + BASELINE_MODELS   # the models that the summaries over models leave out

# ---- figures ------------------------------------------------------------------------------------------------------------
DETAIL_INPUTS = 'main'    # inputs of the models of 11-15 with a figure of every country and model (section 4): 'main'
                          # (those of MAIN_INPUTS that were run: the default run of 11, 13, 14 and 15) or 'all'
MAP           = True      # section 6: maps of the median gain of every country
VS_BASELINE_MODELS = True   # section 7: every model against the baseline models (VGG-16-BN, MLP) in every country

In [ ]:
# ---- the models and the inputs: names, labels and order ---------------------------------------------------------------
TEN_MODELS = ['resnet18', 'resnet34', 'resnet50', 'swin', 'unet', 'vit', 'satclip', 'satclip_only', 'mosaiks',
              'olmoearth']                                       # the models of 11-15, in their order
MODEL_LABEL = {'resnet18': 'ResNet-18', 'resnet34': 'ResNet-34', 'resnet50': 'ResNet-50', 'swin': 'Swin v2-B',
               'unet': 'U-Net', 'vit': 'ViT-S/16', 'satclip': 'ResNet-18 + SatCLIP', 'satclip_only': 'SatCLIP only',
               'mosaiks': 'MOSAIKS', 'olmoearth': 'OlmoEarth', 'vgg16bn': 'VGG-16-BN', 'mlp': 'MLP (band stats)',
               'vgg16': 'VGG-16 (04)',
               'resnet18_finetune': 'ResNet-18 finetune (05)', 'resnet18_frozen': 'ResNet-18 frozen (05)',
               'landsat_resnet18': 'ResNet-18 on Landsat (07)'}
FAMILY_ORDER = ['ten_models', 'table2', 'single']
FAMILY_LABEL = {'ten_models': 'the models of 11-15', 'table2': 'the Table 2 networks of 04 and 05',
                'single': 'the Landsat networks of 07 and 08'}
START_ORDER  = ['pretrained', 'scratch', 'mixed', 'unknown']
MAIN_INPUTS  = ['L+wavelet', 'L+DFT', 'L+peak:pixel11/wavelet6', 'L+TT:cyclic', 'L+NDVI']
SINGLE_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']   # the order of the band cache


def model_label(m):
    """Display name of a model key."""
    if m in MODEL_LABEL:
        return MODEL_LABEL[m]
    if m.startswith('landsat_vits16_'):
        return f'ViT-S/16 on Landsat (08, {m[15:]})'
    return m


OTHER_MODELS = ['vgg16', 'resnet18_finetune', 'resnet18_frozen', 'landsat_resnet18']


def model_order(m):
    """Sort key of a model: the ten models of 11-15 in their order and the baseline models, then those of 04, 05, 07
    and 08, then the others by name."""
    if m in TEN_MODELS:
        return (0, TEN_MODELS.index(m), '')
    if m in BASELINE_MODELS:
        return (0, len(TEN_MODELS) + BASELINE_MODELS.index(m), '')
    if m in OTHER_MODELS:
        return (1, OTHER_MODELS.index(m), '')
    return (2, 0, m)


def combo_order(c):
    """Sort key of an input: the baseline (Landsat), what 11, 13, 14 and 15 add to it, then the other rows of
    04/05 and 07/08."""
    if c == BASELINE:
        return (0, 0, '')
    if not c.startswith(BASELINE + '+'):
        return (12, 0, c)
    tail = c[len(BASELINE) + 1:]
    if tail == 'wavelet':
        return (1, 0, '')
    if tail == 'DFT':
        return (2, 0, '')
    if tail.startswith('peak:'):
        align, _, inp = tail[5:].partition('/')
        aligns, inputs = ['pixel11', 'cluster11', 'calendar'], ['wavelet6', 'ndvi23', 'ndvi23_wavelet6']
        return (3, 10 * (inputs.index(inp) if inp in inputs else 3) + (aligns.index(align) if align in aligns else 3),
                tail)
    if tail.startswith('TT:'):
        return (4, {'cyclic': 0, 'none': 1, 'absolute': 2}.get(tail[3:], 3), tail)
    if tail == 'NDVI':
        return (5, 0, '')
    if tail in SINGLE_BANDS:
        return (6, SINGLE_BANDS.index(tail), '')
    return (7, 0, tail)


def short_combo(c):
    """Short name of an input: what it adds to Landsat ('wavelet', 'DFT', 'peak pixel11/wavelet6', 'TT cyclic',
    'NDVI', 'HFC max', ...), and 'Landsat' for the baseline."""
    if c == BASELINE:
        return 'Landsat'
    tail = c[len(BASELINE) + 1:] if c.startswith(BASELINE + '+') else c
    return tail.replace('peak:', 'peak ').replace('TT:', 'TT ').replace('_', ' ')


def combo_label(c):
    """Long name of an input."""
    if c == BASELINE:
        return 'Landsat (baseline)'
    if not c.startswith(BASELINE + '+'):
        return c.replace('_', ' ')
    tail = c[len(BASELINE) + 1:]
    if tail.startswith('peak:'):
        return f'Landsat + peak-aligned NDVI, {tail[5:]} (14)'
    if tail.startswith('TT:'):
        return f'Landsat + temporal transformer, {tail[3:]} (15)'
    names = {'wavelet': '6 wavelet bands', 'DFT': '5 DFT bands (13)', 'NDVI': '12 monthly NDVI values'}
    return 'Landsat + ' + names.get(tail, tail.replace('_', ' '))


def sort_runs(frame):
    """Rows of a table with the columns start, family, model and combo in the order of START_ORDER, FAMILY_ORDER,
    model_order and combo_order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, FAMILY_ORDER.index(f) if f in FAMILY_ORDER else 9,
             model_order(m), combo_order(c))
            for s, f, m, c in zip(frame['start'], frame['family'], frame['model'], frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)]

In [ ]:
SCORE_COLS = ['n', f'{TARGET} mean', f'{TARGET} sd', 'R2', 'r2', 'RMSE', 'MAE', 'bias']


def cell_scores(y, p, code, n_cells, min_n=None):
    """Scores of the prediction column p in every cell (code: the cell of every cluster, -1 = in none; clusters without
    a prediction are left out). Columns as SCORE_COLS: n, mean and sd of the observed target, R2 against the cell's own
    mean target (negative when the predictions miss the cell's level or spread), r2 (squared correlation of predicted and
    observed, which an offset does not lower; Yeh et al. 2020), RMSE, MAE and bias (mean of predicted - observed), in
    target units (the normalised NL of 02). R2 and r2 need min_n clusters (default MIN_N). Returns an array (n_cells, len(SCORE_COLS))."""
    min_n = MIN_N if min_n is None else min_n
    ok = np.isfinite(p) & (code >= 0)
    c, yy, pp = code[ok], y[ok], p[ok]
    e = pp - yy
    s = lambda w: np.bincount(c, weights=w, minlength=n_cells)[:n_cells]
    n = np.bincount(c, minlength=n_cells)[:n_cells].astype(float)
    sy, syy, sp, spp, spy = s(yy), s(yy * yy), s(pp), s(pp * pp), s(pp * yy)
    se, see, sae = s(e), s(e * e), s(np.abs(e))
    with np.errstate(invalid='ignore', divide='ignore'):
        my = sy / n
        sst = syy - sy * my                       # sum of squares around the cell's mean
        spp_c = spp - sp * sp / n
        cov = spy - sp * my
        enough = (n >= max(min_n, 2)) & (sst > 1e-9)
        r2 = np.where(enough, 1 - see / sst, np.nan)
        rr = np.where(enough & (spp_c > 1e-12), cov ** 2 / (sst * spp_c), np.nan)
        out = np.column_stack([n, my, np.sqrt(sst / (n - 1)), r2, rr, np.sqrt(see / n), sae / n, se / n])
    out[n == 0, 1:] = np.nan
    out[n < 2, 2] = np.nan
    return out


class Bootstrap:
    """Poisson bootstrap over 03's spatial groups (clusters closer than 9.5 km are one group, and a group lies in one
    fold): every resample gives every group a weight drawn from Poisson(1). The same weights serve every comparison, so
    all comparisons are paired, and a subset (a country, its rural clusters) is resampled with the groups it holds."""

    def __init__(self, unit, n_boot, seed):
        self.unit = np.asarray(unit, dtype=np.int64)
        self.n_boot = int(n_boot)
        n_units = int(self.unit.max()) + 1 if len(self.unit) else 0
        rng = np.random.default_rng(seed)
        self.W = rng.poisson(1.0, size=(self.n_boot, n_units)).astype(np.float64) if self.n_boot else None


def _pair_stats(T):
    """R2 of a and b, dR2 = R2(b) - R2(a), dRMSE and dMAE (b - a) from the sums T[..., :7]: n, sum y, sum y^2,
    SSE of a, SSE of b, sum |error of a|, sum |error of b|."""
    n, sy, syy, sa, sb, aa, ab = (T[..., k] for k in range(7))
    with np.errstate(invalid='ignore', divide='ignore'):
        sst = syy - sy * sy / n
        bad = (n < 2) | ~(sst > 1e-9)
        r2a = np.where(bad, np.nan, 1 - sa / sst)
        r2b = np.where(bad, np.nan, 1 - sb / sst)
        return r2a, r2b, r2b - r2a, np.sqrt(sb / n) - np.sqrt(sa / n), (ab - aa) / n


PAIR_COLS = ['n', 'R2 a', 'R2 b', 'dR2', 'dR2 low', 'dR2 high', 'P(dR2 > 0)', 'p', 'dRMSE', 'dRMSE low', 'dRMSE high',
             'dMAE', 'folds better', 'folds compared']


def compare(y, pa, pb, cells, boot, fold_code=None, draws=False):
    """Prediction column b against a on the clusters where both have a prediction, in every cell (name -> positions of
    its clusters). Per cell: n, R2 of a and b, dR2 = R2(b) - R2(a) (a and b share the clusters, so this is their
    difference in squared error over the cell's sum of squares), its 95 % bootstrap interval, P(dR2 > 0) and the
    two-sided p of the resamples, dRMSE and dMAE (b - a in target units; negative = b better), and in how many test folds
    b has the higher R2 (of the folds with at least MIN_FOLD_N of the cell's clusters). A cell with fewer than MIN_N
    common clusters gets n only. Returns {cell: dict with the keys of PAIR_COLS}; draws=True adds 'draws', the dR2 of
    every resample (the same resamples for every cell and comparison, so differences of draws are paired)."""
    both = np.isfinite(pa) & np.isfinite(pb)
    ea, eb = y - pa, y - pb
    Q = np.column_stack([np.ones(len(y)), y, y * y, ea * ea, eb * eb, np.abs(ea), np.abs(eb)])
    out = {}
    for name, pos in cells.items():
        pos = pos[both[pos]]
        rec = {'n': int(len(pos))}
        out[name] = rec
        if len(pos) < MIN_N:
            continue
        r2a, r2b, d, drmse, dmae = (float(v) for v in _pair_stats(Q[pos].sum(axis=0)))
        rec.update({'R2 a': r2a, 'R2 b': r2b, 'dR2': d, 'dRMSE': drmse, 'dMAE': dmae})
        if boot is not None and boot.n_boot:
            u, inv = np.unique(boot.unit[pos], return_inverse=True)
            inv = inv.ravel()
            S = np.column_stack([np.bincount(inv, weights=Q[pos, k], minlength=len(u)) for k in range(7)])
            _, _, bd, brmse, _ = _pair_stats(boot.W[:, u] @ S)
            ok = np.isfinite(bd)
            if draws:
                rec['draws'] = bd
            if ok.sum() >= 10:
                b = bd[ok]
                rec.update({'dR2 low': float(np.quantile(b, 0.025)), 'dR2 high': float(np.quantile(b, 0.975)),
                            'P(dR2 > 0)': float((b > 0).mean()),
                            'p': float(min(1.0, 2 * (1 + min((b <= 0).sum(), (b >= 0).sum())) / (len(b) + 1))),
                            'dRMSE low': float(np.nanquantile(brmse[ok], 0.025)),
                            'dRMSE high': float(np.nanquantile(brmse[ok], 0.975))})
        if fold_code is not None:
            f = fold_code[pos]
            keep = f >= 0
            Sf = np.column_stack([np.bincount(f[keep], weights=Q[pos[keep], k], minlength=len(FOLDS))[:len(FOLDS)]
                                  for k in range(7)])
            use = Sf[:, 0] >= MIN_FOLD_N
            fd = _pair_stats(Sf[use])[2]
            rec['folds better'] = int((fd > 0).sum())
            rec['folds compared'] = int(np.isfinite(fd).sum())
    return out


def bh(p):
    """Benjamini-Hochberg q-values of the p-values p (NaN stays NaN)."""
    p = np.asarray(p, dtype=float)
    q = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if len(ok):
        order = ok[np.argsort(p[ok], kind='stable')]
        ranked = p[order] * len(ok) / np.arange(1, len(ok) + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def add_q(frame, keys, where=None, split=None):
    """Column 'q': the Benjamini-Hochberg q-values of 'p' within every group of keys (rows of `where` only); `split`
    (one label per row, e.g. baseline_split) divides the groups further."""
    frame = frame.copy()
    frame['q'] = np.nan
    rows = frame.index if where is None else frame.index[np.asarray(where, bool)]
    by = list(keys)
    if split is not None:
        frame['_split'] = np.asarray(split)
        by.append('_split')
    for _, g in frame.loc[rows].groupby(by, sort=False):
        frame.loc[g.index, 'q'] = bh(g['p'].to_numpy(float))
    return frame.drop(columns='_split') if split is not None else frame


def baseline_split(frame):
    """'baseline' for the rows of a baseline model (BASELINE_MODELS; in a table of pairs: either model), 'main' for all
    others: add_q(..., split=baseline_split(frame)) gives the comparisons with a baseline model their own q-values, so
    those of the other models stay the same whether or not the baselines have run."""
    is_base = frame['model'].isin(BASELINE_MODELS).to_numpy()
    if 'other model' in frame:
        is_base = is_base | frame['other model'].isin(BASELINE_MODELS).to_numpy()
    return np.where(is_base, 'baseline', 'main')


def verdict(low, high):
    """'better' when the 95 % interval of a difference lies above 0, 'worse' below 0, 'n.s.' when it holds 0."""
    low, high = np.asarray(low, dtype=float), np.asarray(high, dtype=float)
    return np.where(low > 0, 'better', np.where(high < 0, 'worse', np.where(np.isfinite(low), 'n.s.', '')))


def fmt_delta(v, digits=3):
    """A signed difference without the leading zero: 0.0123 -> '+.012', -1.5 -> '-1.500', NaN -> '–'."""
    if v is None or not np.isfinite(v):
        return '–'
    return re.sub(r'^([+-])0\.', r'\1.', f'{v:+.{digits}f}')


def fmt_interval(d, low, high, digits=3):
    """'+.012 [+.004, +.020]' (without the interval when there is none)."""
    if d is None or not np.isfinite(d):
        return '–'
    if low is None or not np.isfinite(low):
        return fmt_delta(d, digits)
    return f'{fmt_delta(d, digits)} [{fmt_delta(low, digits)}, {fmt_delta(high, digits)}]'


def write_csv(frame, path, **kw):
    """Write a CSV file (gzip for .gz) under a temporary name, then rename it, so nobody reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    if path.endswith('.gz'):
        kw.setdefault('compression', 'gzip')
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)

In [ ]:
# Colors of the reference palette of the data-viz guide: chart ink, a blue -> red diverging scale with a gray midpoint
# for differences (blue: better than the reference), a one-hue blue scale for R2, and categorical slots 1 and 2 (blue,
# orange; checked for colour-vision deficiency) for rural and urban, with their own marker shapes.
INK, SECONDARY, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
SERIES_1    = '#2a78d6'                                    # categorical slot 1: a single series
AREA_COLOR  = {'rural': '#2a78d6', 'urban': '#eb6834'}
AREA_MARKER = {'rural': 'o', 'urban': 's'}
SEQUENTIAL  = LinearSegmentedColormap.from_list('r2_blue', ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf',
                                                            '#184f95', '#0d366b'])
DIVERGING   = LinearSegmentedColormap.from_list('delta', ['#e34948', '#f0efec', '#2a78d6'])
for _cmap in (SEQUENTIAL, DIVERGING):
    _cmap.set_bad(SURFACE)
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'axes.edgecolor': AXIS, 'axes.labelcolor': SECONDARY, 'axes.titlecolor': INK,
                     'axes.titlesize': 10, 'axes.titlelocation': 'left', 'axes.labelsize': 9, 'font.size': 9,
                     'xtick.color': SECONDARY, 'ytick.color': SECONDARY, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'text.color': INK, 'legend.frameon': False, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False, 'grid.color': GRID,
                     'grid.linewidth': 0.8})


def text_color(rgba):
    """White on dark cells, ink on light ones."""
    r, g, b = rgba[:3]
    return 'white' if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.45 else INK


def delta_norm(values, floor=0.01, q=0.95):
    """A symmetric scale for differences: +-(the q quantile of |values|, at least floor), rounded up to 0.005."""
    v = np.abs(np.asarray(values, dtype=float))
    v = v[np.isfinite(v)]
    top = max(floor, float(np.quantile(v, q)) if len(v) else floor)
    top = np.ceil(top / 0.005) * 0.005
    return Normalize(-top, top)


def heatmap(ax, values, row_labels, col_labels, cmap, norm, labels=None, marks=None, bold=None, fontsize=6.5):
    """Coloured cells (rows x cols) with a thin surface gap. labels: text per cell (None: none); marks: cells that get a
    small dot (a significant difference, when there is no text); bold: cells whose text is bold. NaN cells get a muted
    dash."""
    values = np.asarray(values, dtype=float)
    im = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_xticks(np.arange(-0.5, len(col_labels)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels)), minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=1.5)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            if not np.isfinite(v):
                ax.text(j, i, '–', ha='center', va='center', fontsize=fontsize, color=MUTED)
                continue
            color = text_color(cmap(norm(v)))
            if labels is not None and labels[i][j]:
                ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=fontsize, color=color,
                        fontweight='bold' if bold is not None and bold[i][j] else 'normal')
            elif marks is not None and marks[i][j]:
                ax.plot(j, i, marker='o', markersize=2.6, color=color, linestyle='none')
    return im


def forest(ax, y, d, low, high, color, marker='o', label=None):
    """Differences d with their 95 % intervals at the heights y: a filled marker where the interval excludes 0, an open
    one where it holds 0. Returns a handle for the legend."""
    d, low, high, y = (np.asarray(v, dtype=float) for v in (d, low, high, y))
    has = np.isfinite(low) & np.isfinite(high)
    ax.hlines(y[has], low[has], high[has], color=color, linewidth=1.4)
    sig = has & ((low > 0) | (high < 0))
    ok = np.isfinite(d)
    ax.scatter(d[ok & sig], y[ok & sig], s=34, marker=marker, color=color, edgecolors=SURFACE, linewidths=1.0,
               zorder=3)
    ax.scatter(d[ok & ~sig], y[ok & ~sig], s=34, marker=marker, facecolors=SURFACE, edgecolors=color,
               linewidths=1.3, zorder=3)
    return Line2D([], [], marker=marker, linestyle='none', color=color, markersize=6, label=label)


FILLED_OPEN = [Line2D([], [], marker='o', linestyle='none', color=SECONDARY, markersize=6,
                      label='filled: the 95 % interval excludes 0'),
               Line2D([], [], marker='o', linestyle='none', markerfacecolor=SURFACE, markeredgecolor=SECONDARY,
                      markersize=6, label='open: it holds 0')]


def save_figure(fig, name):
    """Save a figure as EVAL_DIR/<name>.png (written under a temporary name, then renamed) and show that file, so the
    notebook shows exactly the saved plot."""
    path = os.path.join(EVAL_DIR, f'{name}.png')
    fig.savefig(path + '.tmp.png', dpi=150, bbox_inches='tight')
    os.replace(path + '.tmp.png', path)
    plt.close(fig)
    display(Image(filename=path))
    print('->', path)


def grid_of(n, ncols):
    """Rows and columns of a grid of n panels with at most ncols columns, as even as possible (4 panels: 2 x 2)."""
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    return nrows, int(np.ceil(n / nrows))

In [ ]:
def consensus(table, keys, extras=True):
    """Over the models of every group of keys (without SUMMARY_SKIP: the noise and the baseline models): models, median
    and mean dR2, models with dR2 > 0, models significantly better / worse (interval above / below 0), the model with
    the largest gain; the mean dR2 of the noise models (training noise) and the dR2 of every baseline model. With
    extras=False (the comparisons with a baseline model) the model with the smallest dR2 instead of the last two."""
    out = []
    baselines = [b for b in BASELINE_MODELS if b in set(table['model'])]
    for k, g in table.groupby(keys, sort=False):
        real = g[~g['model'].isin(SUMMARY_SKIP) & g['dR2'].notna()]
        noise = g[g['model'].isin(NOISE_MODELS) & g['dR2'].notna()]
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        rec.update({'n': int(g['n'].max()), 'models': len(real),
                    'median dR2': real['dR2'].median() if len(real) else np.nan,
                    'mean dR2': real['dR2'].mean() if len(real) else np.nan,
                    'models better': int((real['dR2'] > 0).sum()),
                    'sig. better': int((real['verdict'] == 'better').sum()),
                    'sig. worse': int((real['verdict'] == 'worse').sum()),
                    'best model': model_label(real.loc[real['dR2'].idxmax(), 'model']) if len(real) else '',
                    'best dR2': real['dR2'].max() if len(real) else np.nan})
        if extras:
            rec.update({'noise dR2': noise['dR2'].mean() if len(noise) else np.nan,
                        **{f'{b} dR2': g.loc[g['model'] == b, 'dR2'].mean() for b in baselines}})
        else:
            rec.update({'worst model': model_label(real.loc[real['dR2'].idxmin(), 'model']) if len(real) else '',
                        'worst dR2': real['dR2'].min() if len(real) else np.nan})
        out.append(rec)
    out = pd.DataFrame(out)
    return sort_runs(out.assign(model='')).drop(columns='model').reset_index(drop=True) if len(out) else out


def consensus_cell(r):
    """'+.012 (8/9 ▲3 ▼1)': median dR2, models better / models, significantly better (▲) and worse (▼)."""
    if not r['models'] or not np.isfinite(r['median dR2']):
        return ''
    return (f"{fmt_delta(r['median dR2'])} ({r['models better']}/{r['models']}"
            + (f" ▲{r['sig. better']}" if r['sig. better'] else '') + (f" ▼{r['sig. worse']}" if r['sig. worse'] else '')
            + ')')

In [ ]:
# ---- every model of 11-15 against the baseline models (BASELINE_MODELS: VGG-16-BN and the MLP on band statistics) ------
BM_AGAINST = ['same input', 'Landsat']   # the run of the baseline model that a run is compared with: the one with
                                         # the same input (what the architecture adds over the baseline model) and the
                                         # one with Landsat (what the architecture and the added bands add together
                                         # over the baseline model on the baseline input)
BM_COLS = ['start', 'family', 'model', 'combo', 'baseline model', 'against', 'baseline run']


def baseline_model_pairs(frame):
    """(run, run of the baseline model, baseline model, against) for every run of the models of 11-15 in frame and every
    baseline model with a run of the same start: against its run with the same input ('same input'; not for the
    baseline model itself) and against its run with Landsat ('Landsat'; for the baseline model itself this is
    its own gain from the input, as in the comparisons with Landsat)."""
    t = frame[frame['family'] == 'ten_models']
    rid = {(r.model, r.combo, r.start): r.run_id for r in t.itertuples()}
    out = []
    for r in t.itertuples():
        for b in BASELINE_MODELS:
            same, base = rid.get((b, r.combo, r.start)), rid.get((b, BASELINE, r.start))
            if same is not None and r.model != b:
                out.append((r.run_id, same, b, 'same input'))
            if base is not None and base != r.run_id:
                out.append((r.run_id, base, b, 'Landsat'))
    return out


def sort_bm(frame):
    """Rows in the order of start, baseline model, reference (against), model and input; rows that share these (the
    countries, the areas) keep their order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, BASELINE_MODELS.index(b) if b in BASELINE_MODELS else 9,
             BM_AGAINST.index(a) if a in BM_AGAINST else 9, model_order(m), combo_order(c))
            for s, b, a, m, c in zip(frame['start'], frame['baseline model'], frame['against'], frame['model'],
                                     frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)].reset_index(drop=True)


def bm_label(b, against):
    """'VGG-16-BN, same input' or 'VGG-16-BN on Landsat': a baseline model and the run it is compared with."""
    return f'{model_label(b)}, same input' if against == 'same input' else f'{model_label(b)} on Landsat'


def bm_summary(table, keys):
    """consensus() of a table of comparisons with the baseline models over the ten models (without SUMMARY_SKIP), for
    every start, baseline model, reference and the other keys, in the order of sort_bm."""
    if not len(table):
        return pd.DataFrame(columns=['start', 'family', 'baseline model', 'against'] + list(keys) +
                            ['n', 'models', 'median dR2', 'mean dR2', 'models better', 'sig. better', 'sig. worse',
                             'best model', 'best dR2', 'worst model', 'worst dR2'])
    out = consensus(table.assign(family='ten_models'), ['start', 'family', 'baseline model', 'against'] + list(keys),
                    extras=False)
    return sort_bm(out.assign(model='')).drop(columns='model')

# 1. The collected test predictions

The runs that 16 used, with their family, model, input and start, and the test prediction of every cluster.

In [ ]:
RUNS_CSV = os.path.join(COLLECTED_DIR, 'eval_runs.csv')
PRED_CSV = os.path.join(COLLECTED_DIR, 'eval_test_predictions.csv.gz')
assert os.path.exists(RUNS_CSV) and os.path.exists(PRED_CSV), \
    f'{RUNS_CSV} or {PRED_CSV} is missing: run 16_Evaluate_All_Models first (same COLLECTED_DIR)'

runs = pd.read_csv(RUNS_CSV, dtype={'folds': str, 'notebook': str}, keep_default_na=False,
                   na_values={'n_test': [''], 'n_folds': ['']})
runs['used'] = runs['used'].astype(str).str.lower().eq('true')
runs = sort_runs(runs[runs['used']]).set_index('run_id', drop=False)
assert len(runs), f'no used run in {RUNS_CSV}: run 16 again when runs of 04-15 have finished'
wide = pd.read_csv(PRED_CSV, dtype={'cluster_id': str, 'country': str, 'area': str, 'fold': str, 'nl_sensor': str})
missing = [r for r in runs['run_id'] if r not in wide.columns]
assert not missing, f'runs of eval_runs.csv without predictions in {PRED_CSV}: {missing[:5]}'
if NL_PERIOD is not None:                                         # only the clusters of one NL period (one scale)
    assert NL_PERIOD in ('DMSP', 'VIIRS'), f"NL_PERIOD must be None, 'DMSP' or 'VIIRS', not {NL_PERIOD!r}"
    wide = wide[wide['nl_sensor'].astype(str).eq(NL_PERIOD).to_numpy()].reset_index(drop=True)
    few = [r for r in runs['run_id'] if wide[r].notna().sum() < MIN_N]
    runs = runs.drop(index=few)
    print(f'NL_PERIOD = {NL_PERIOD}: {len(wide)} clusters of that period; {len(few)} runs with fewer than {MIN_N} of '
          f'them left out' + (f' ({", ".join(few[:3])}{", ..." if len(few) > 3 else ""})' if few else ''))
    assert len(wide) and len(runs), f'no run has {MIN_N} test clusters of the {NL_PERIOD} period'

N = len(wide)
Y = wide[TARGET].to_numpy(dtype=float)
P = {r: wide[r].to_numpy(dtype=float) for r in runs['run_id']}
COUNTRY = wide['country'].fillna('').astype(str).to_numpy()
AREA = wide['area'].fillna('').astype(str).to_numpy()
COUNTRIES = sorted(set(COUNTRY) - {''})
COUNTRY_CODE = pd.Categorical(COUNTRY, categories=COUNTRIES).codes.astype(np.int64)
FOLD_CODE = pd.Categorical(wide['fold'].fillna('').astype(str), categories=FOLDS).codes.astype(np.int64)
BOOT = Bootstrap(pd.factorize(wide['group'])[0], BOOTSTRAP, SEED)
CONTINENT = f'all {len(COUNTRIES)} countries' + (f', {NL_PERIOD} clusters' if NL_PERIOD else '')
N_COUNTRY = pd.Series(COUNTRY).value_counts()
os.makedirs(EVAL_DIR, exist_ok=True)

print(f'{len(runs)} runs with test predictions for {N} clusters in {len(COUNTRIES)} countries ({RUNS_CSV})')
print('runs per start and family:')
display(runs.pivot_table(index='start', columns='family', values='run_id', aggfunc='count', fill_value=0)
        .reindex(index=[s for s in START_ORDER if s in set(runs['start'])],
                 columns=[f for f in FAMILY_ORDER if f in set(runs['family'])]))
print(f'bootstrap: {BOOTSTRAP} resamples of {BOOT.W.shape[1] if BOOT.n_boot else 0} spatial groups')


def baseline_pairs(frame):
    """(run, the run of its baseline input) for every run of frame whose input is not the baseline and whose family,
    model and start have a baseline run."""
    base = {(r.family, r.model, r.start): r.run_id for r in frame.itertuples() if r.combo == BASELINE}
    return [(r.run_id, base[(r.family, r.model, r.start)]) for r in frame.itertuples()
            if r.combo != BASELINE and (r.family, r.model, r.start) in base]


def run_info(run_id):
    """The columns of eval_runs.csv that describe a run."""
    r = runs.loc[run_id]
    return {'start': r['start'], 'family': r['family'], 'model': r['model'], 'combo': r['combo'],
            'notebook': r['notebook']}


def country_label(c):
    """'Nigeria (3759)': the country and its number of test clusters."""
    return f'{c} ({N_COUNTRY.get(c, 0)})'

# 2. Scores of every run in every country

`country_metrics.csv` holds the scores of every run in every country (and on the whole continent, `country` = `all ... countries`). The table below shows the R² of the models of 11–15 (the ten models and the baseline models VGG-16-BN and MLP) with the baseline input, Landsat, per country, for every start; the plot shows the same.

In [ ]:
rows = []
ZERO = np.zeros(N, dtype=np.int64)
for r in runs.itertuples():
    per = cell_scores(Y, P[r.run_id], COUNTRY_CODE, len(COUNTRIES))
    cont = cell_scores(Y, P[r.run_id], ZERO, 1)[0]
    for c, v in zip([CONTINENT] + COUNTRIES, np.vstack([cont, per])):
        if v[0] > 0:
            rows.append({**run_info(r.run_id), 'country': c, **dict(zip(SCORE_COLS, v)), 'run_id': r.run_id})
country_metrics = pd.DataFrame(rows)
write_csv(country_metrics, os.path.join(EVAL_DIR, 'country_metrics.csv'))
print(f'{len(country_metrics)} rows (run x country) ->', os.path.join(EVAL_DIR, 'country_metrics.csv'))

base_r2 = country_metrics[(country_metrics['family'] == 'ten_models') & (country_metrics['combo'] == BASELINE)]
for start in [s for s in START_ORDER if s in set(base_r2['start'])]:
    t = base_r2[base_r2['start'] == start].pivot(index='country', columns='model', values='R2')
    t = t.reindex(index=[CONTINENT] + [c for c in COUNTRIES if c in t.index],
                  columns=sorted(t.columns, key=model_order))
    t.insert(0, 'n', N_COUNTRY.reindex(t.index).fillna(N).astype(int).to_numpy())
    print(f'\n{start.upper()}: R² of the models of 11-15 with Landsat, per country (against its own mean target)')
    display(t.rename(columns=model_label).round(3))

In [ ]:
def r2_by_country(table, name, title):
    """R2 per country (rows) and model (columns), one panel per start."""
    starts = [s for s in START_ORDER if s in set(table['start'])]
    if not starts:
        print('no runs for', name)
        return
    models = sorted(set(table['model']), key=model_order)
    countries = [c for c in COUNTRIES if c in set(table['country'])]
    fig, axes = plt.subplots(1, len(starts), figsize=(0.55 * len(models) * len(starts) + 3.2,
                                                      0.26 * len(countries) + 2.6),
                             squeeze=False, constrained_layout=True)
    norm = Normalize(0, 1)
    for ax, start in zip(axes[0], starts):
        m = table[table['start'] == start].pivot(index='country', columns='model', values='R2').reindex(
            index=countries, columns=models).to_numpy(dtype=float)
        im = heatmap(ax, m, [country_label(c) for c in countries], [model_label(k) for k in models], SEQUENTIAL, norm,
                     labels=[[f'{x:.2f}' if np.isfinite(x) else '' for x in row] for row in m], fontsize=5.5)
        ax.set_title(start, fontsize=10)
        if ax is not axes[0][0]:
            ax.set_yticklabels([])
    fig.colorbar(im, ax=list(axes[0]), shrink=0.6, extend='min', label='R² within the country')
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    save_figure(fig, name)


r2_by_country(base_r2[base_r2['country'] != CONTINENT], 'country_r2_baseline',
              'R² of the models of 11-15 with Landsat in every country (test clusters in brackets)')

# 3. Every input against Landsat, per country

For every run whose family, model and start have a baseline run, the paired comparison with that baseline in every country (and on the whole continent). `country_vs_baseline.csv` has every model; the tables below summarise over the ten models (`country_consensus.csv`, without `satclip_only` and the baseline models, whose ΔR² it shows on their own): **median ΔR² (models better / models; ▲ significantly better, ▼ significantly worse)**. A country where most models gain, and several with an interval above 0, is one where the added NDVI information helps; a country where they split or the median is near the `noise dR2` of `satclip_only` shows no reliable effect.

In [ ]:
PAIRS = baseline_pairs(runs)
if not PAIRS:
    print('no run has an input other than Landsat together with its baseline run yet (11, 13, 14, 15; the '
          'other rows of 04/05 and 07/08): the comparisons below stay empty')
CELLS = {CONTINENT: np.arange(N), **{c: np.flatnonzero(COUNTRY == c) for c in COUNTRIES}}
t0 = time.time()
rows = []
for run_id, base_id in PAIRS:
    res = compare(Y, P[base_id], P[run_id], CELLS, BOOT, FOLD_CODE)
    for cell, rec in res.items():
        rows.append({**run_info(run_id), 'country': cell, 'baseline run': base_id,
                     **{k: rec.get(k, np.nan) for k in PAIR_COLS}})
vs = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'country', 'baseline run'] + PAIR_COLS)
vs = vs.rename(columns={'R2 a': 'R2 baseline', 'R2 b': 'R2'})
vs = add_q(vs, ['start', 'family', 'model', 'combo'], where=(vs['country'] != CONTINENT) & vs['p'].notna())
vs['verdict'] = verdict(vs['dR2 low'], vs['dR2 high'])
vs = sort_runs(vs).reset_index(drop=True)
write_csv(vs, os.path.join(EVAL_DIR, 'country_vs_baseline.csv'))
print(f'{len(PAIRS)} runs against their baseline in {len(CELLS)} cells in {time.time() - t0:.0f} s ->',
      os.path.join(EVAL_DIR, 'country_vs_baseline.csv'))


cons = consensus(vs, ['start', 'family', 'combo', 'country'])
write_csv(cons, os.path.join(EVAL_DIR, 'country_consensus.csv'))

for start in [s for s in START_ORDER if len(cons) and s in set(cons['start'])]:
    for fam in [f for f in FAMILY_ORDER if f in set(cons.loc[cons['start'] == start, 'family'])]:
        t = cons[(cons['start'] == start) & (cons['family'] == fam)]
        m = t.assign(cell=t.apply(consensus_cell, axis=1)).pivot(index='country', columns='combo', values='cell')
        m = m.reindex(index=[CONTINENT] + [c for c in COUNTRIES if c in m.index],
                      columns=sorted(m.columns, key=combo_order)).fillna('')
        m.index = [i if i == CONTINENT else country_label(i) for i in m.index]
        print(f'\n{start.upper()}, {FAMILY_LABEL[fam]}: median ΔR² against Landsat over the models '
              f'(models better / models; ▲ significantly better, ▼ worse)')
        display(m.rename(columns=short_combo))

# 4. Figures per country

* `country_consensus_<start>.png`: the median ΔR² over the ten models (without `satclip_only` and the baseline models) for every country and input, with the number of models that gain.
* `country_vs_baseline_<start>_ten_models.png`: ΔR² of every model and country, one panel per input (`DETAIL_INPUTS`); a dot marks an interval that excludes 0.
* `country_vs_baseline_<start>_<family>.png` for the Table 2 networks and the Landsat networks of 07 and 08: one panel per model, every input.

Blue: the input is better than Landsat; red: worse. The colour scale is cut at the 95th percentile of |ΔR²| of the plot; the values are in the CSV files.

In [ ]:
def consensus_figure(start):
    t = cons[(cons['start'] == start) & (cons['family'] == 'ten_models') & (cons['country'] != CONTINENT)]
    if not len(t):
        return
    combos = sorted(set(t['combo']), key=combo_order)
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    med = t.pivot(index='country', columns='combo', values='median dR2').reindex(index=countries, columns=combos)
    lab = t.assign(cell=[f"{fmt_delta(d)}\n{b}/{m}" if m else '' for d, b, m in
                         zip(t['median dR2'], t['models better'], t['models'])]).pivot(
        index='country', columns='combo', values='cell').reindex(index=countries, columns=combos).fillna('')
    fig, ax = plt.subplots(figsize=(0.95 * len(combos) + 3.4, 0.36 * len(countries) + 2.4), constrained_layout=True)
    im = heatmap(ax, med.to_numpy(dtype=float), [country_label(c) for c in countries], [short_combo(c) for c in combos],
                 DIVERGING, delta_norm(med.to_numpy(dtype=float)), labels=lab.to_numpy().tolist(), fontsize=5.5)
    ax.xaxis.tick_top()
    ax.tick_params(axis='x', labelrotation=90)
    fig.colorbar(im, ax=ax, shrink=0.6, extend='both', label='median ΔR² over the models (input − Landsat)')
    ax.set_title(f'{start.capitalize()}: the ten models, median ΔR² against Landsat per country\n'
                 '(text: median ΔR², models better / models; without satclip_only and the baseline models)', fontsize=10,
                 pad=12)
    save_figure(fig, f'country_consensus_{start}')


def detail_figure(table, start, fam, panel_key, col_key, name, title, panels=None,
                  cbar_label='ΔR² against Landsat within the country (dot: 95 % interval excludes 0)'):
    """ΔR² of every country (rows) and col_key (columns), one panel per value of panel_key; dots: interval excludes 0."""
    t = table[(table['start'] == start) & (table['family'] == fam) & (table['country'] != CONTINENT)]
    order = {'combo': combo_order, 'model': model_order}
    label = {'combo': short_combo, 'model': model_label}
    panels = [p for p in (panels or sorted(set(t[panel_key]), key=order[panel_key])) if p in set(t[panel_key])]
    if not panels:
        return
    cols = sorted(set(t[col_key]), key=order[col_key])
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    norm = delta_norm(t.loc[~t['model'].isin(NOISE_MODELS), 'dR2'])
    nr, nc = grid_of(len(panels), 4)
    fig, axes = plt.subplots(nr, nc, figsize=((0.3 * len(cols) + 0.4) * nc + 3.0, (0.22 * len(countries) + 1.6) * nr + 0.6),
                             squeeze=False, sharey=True, constrained_layout=True)
    for ax, p in zip(axes.flat, panels):
        g = t[t[panel_key] == p]
        m = g.pivot(index='country', columns=col_key, values='dR2').reindex(index=countries, columns=cols)
        s = g.assign(s=g['verdict'].isin(['better', 'worse']).astype(float)).pivot(
            index='country', columns=col_key, values='s').reindex(index=countries, columns=cols).fillna(0) > 0.5
        im = heatmap(ax, m.to_numpy(dtype=float), [country_label(c) for c in countries],
                     [label[col_key](k) for k in cols], DIVERGING, norm, marks=s.to_numpy(dtype=bool), fontsize=5)
        ax.set_title(label[panel_key](p), fontsize=9)
    for ax in axes.flat[len(panels):]:
        ax.axis('off')
    fig.colorbar(im, ax=list(axes.flat), shrink=0.5, extend='both', label=cbar_label)
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    save_figure(fig, name)


for start in [s for s in START_ORDER if s in set(vs['start'])]:
    consensus_figure(start)
    tv = vs[(vs['start'] == start) & (vs['family'] == 'ten_models')]
    if len(tv):
        detail = sorted(set(tv['combo']), key=combo_order) if DETAIL_INPUTS == 'all' else \
            [c for c in MAIN_INPUTS if c in set(tv['combo'])] or sorted(set(tv['combo']), key=combo_order)
        detail_figure(vs, start, 'ten_models', 'combo', 'model', f'country_vs_baseline_{start}_ten_models',
                      f'{start.capitalize()}: every model of 11-15 against its Landsat run, per country',
                      panels=detail)
    for fam in ('table2', 'single'):
        if len(vs[(vs['start'] == start) & (vs['family'] == fam)]):
            detail_figure(vs, start, fam, 'model', 'combo', f'country_vs_baseline_{start}_{fam}',
                          f'{start.capitalize()}: {FAMILY_LABEL[fam]}, every row against Landsat, per country')

# 5. The best input per country

For every model, start and country: the input with the highest R² (the baseline included) and its gain over Landsat (`country_best_input.csv`). The tables show, per country, the input with the largest median ΔR² over the ten models and how many models gain with it; `Landsat` when no input has a positive median.

In [ ]:
best = []
for (start, fam, model, country), g in country_metrics.groupby(['start', 'family', 'model', 'country'], sort=False):
    g = g[g['R2'].notna()]
    if not len(g) or BASELINE not in set(g['combo']):
        continue
    b = g.loc[g['R2'].idxmax()]
    r2_base = float(g.loc[g['combo'] == BASELINE, 'R2'].iloc[0])
    best.append({'start': start, 'family': fam, 'model': model, 'combo': b['combo'], 'country': country,
                 'n': int(b['n']), 'best R2': b['R2'], 'baseline R2': r2_base, 'gain': b['R2'] - r2_base,
                 'inputs compared': len(g)})
best = pd.DataFrame(best)
if len(best):
    best = sort_runs(best).reset_index(drop=True)
    best = best.rename(columns={'combo': 'best input'})
write_csv(best, os.path.join(EVAL_DIR, 'country_best_input.csv'))

for start in [s for s in START_ORDER if len(cons) and s in set(cons['start'])]:
    t = cons[(cons['start'] == start) & (cons['family'] == 'ten_models')]
    if not len(t):
        continue
    out = []
    for country in [CONTINENT] + [c for c in COUNTRIES if c in set(t['country'])]:
        g = t[(t['country'] == country) & t['median dR2'].notna()]
        if not len(g):
            continue
        top = g.loc[g['median dR2'].idxmax()]
        positive = top['median dR2'] > 0
        out.append({'country': country if country == CONTINENT else country_label(country),
                    'best input': short_combo(top['combo']) if positive else 'Landsat',
                    'median dR2': top['median dR2'] if positive else np.nan,
                    'models better': f"{top['models better']}/{top['models']}" if positive else '',
                    'sig. better': top['sig. better'] if positive else np.nan,
                    'inputs with a positive median': int((g['median dR2'] > 0).sum()),
                    'inputs compared': len(g)})
    print(f'\n{start.upper()}, the ten models: the input with the largest median gain over Landsat per country')
    display(pd.DataFrame(out).round(4))
    bt = best[(best['start'] == start) & (best['family'] == 'ten_models') & ~best['model'].isin(NOISE_MODELS)] \
        if len(best) else best
    if len(bt):
        m = bt.assign(cell=bt['best input'].map(short_combo)).pivot(index='country', columns='model', values='cell')
        m = m.reindex(index=[CONTINENT] + [c for c in COUNTRIES if c in m.index], columns=sorted(m.columns, key=model_order))
        print(f'{start.upper()}: the input with the highest R² for every model and country (without {NOISE_MODELS})')
        display(m.rename(columns=model_label).fillna(''))

# 6. Map of the gains

Every test cluster coloured by the median ΔR² of its country over the ten models (one panel per input of `DETAIL_INPUTS`): where in Africa an input helps the models within the countries. Grey: no comparison.

In [ ]:
if MAP and {'lat', 'lon'} <= set(wide.columns):
    lat, lon = wide['lat'].to_numpy(dtype=float), wide['lon'].to_numpy(dtype=float)
    for start in [s for s in START_ORDER if len(cons) and s in set(cons['start'])]:
        t = cons[(cons['start'] == start) & (cons['family'] == 'ten_models') & (cons['country'] != CONTINENT)]
        if not len(t):
            continue
        combos = sorted(set(t['combo']), key=combo_order) if DETAIL_INPUTS == 'all' else \
            [c for c in MAIN_INPUTS if c in set(t['combo'])] or sorted(set(t['combo']), key=combo_order)
        norm = delta_norm(t['median dR2'])
        nr, nc = grid_of(len(combos), 3)
        fig, axes = plt.subplots(nr, nc, figsize=(4.6 * nc + 1.2, 4.8 * nr + 0.5), squeeze=False,
                                 constrained_layout=True)
        ok = np.isfinite(lat) & np.isfinite(lon)
        for ax, combo in zip(axes.flat, combos):
            value = pd.Series(COUNTRY).map(t[t['combo'] == combo].set_index('country')['median dR2']).to_numpy(float)
            has = ok & np.isfinite(value)
            ax.scatter(lon[ok], lat[ok], s=5, color=AXIS, linewidths=0, rasterized=True)   # every cluster
            ax.scatter(lon[ok & ~has], lat[ok & ~has], s=2, color=GRID, linewidths=0, rasterized=True)
            sc = ax.scatter(lon[has], lat[has], s=2, c=value[has], cmap=DIVERGING, norm=norm, linewidths=0,
                            rasterized=True)
            ax.set_aspect(1 / np.cos(np.radians(np.nanmedian(lat[ok]))))
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_title(short_combo(combo), fontsize=9)
        for ax in axes.flat[len(combos):]:
            ax.axis('off')
        fig.colorbar(sc, ax=list(axes.flat), shrink=0.6, extend='both',
                     label='median ΔR² of the country over the ten models')
        fig.suptitle(f'{start.capitalize()}: the gain of every input over Landsat within each country',
                     x=0.01, ha='left', fontsize=11)
        save_figure(fig, f'country_map_{start}')
else:
    print('no map (MAP = False, or no lat/lon in the collected predictions)')

# 7. Every model against the baseline models, per country

The baseline models of 11–15 (`BASELINE_MODELS`: VGG-16-BN and the MLP on band statistics) as the reference for every model of 11–15 in every country, with the same start, in the two ways of 16:

* **same input**: ΔR² = R²(model with input I) − R²(baseline model with input I) within the country: what the architecture adds over the baseline model;
* **Landsat**: ΔR² = R²(model with input I) − R²(baseline model with Landsat): what the architecture and the added bands add together (for the baseline model itself: its own gain from the input).

`country_vs_baseline_models.csv` holds every model, input, country, baseline model and reference (`q`: Benjamini–Hochberg over the countries of one model, input, start, baseline model and reference); `country_vs_baseline_models_summary.csv` the summary over the ten models (without `satclip_only` and the baseline models), with the weakest model. The tables: **median ΔR² over the ten models (models better / models; ▲ significantly better, ▼ worse)**, one per start, baseline model and reference. The figures: `country_vs_baseline_models_<start>_<baseline model>.png`, this median for every country and input (left: same input, right: on Landsat), and `country_vs_<baseline model>_<start>.png`, every model and country with the same input, one panel per input (Landsat and `DETAIL_INPUTS`).

In [ ]:
BM_PAIRS = baseline_model_pairs(runs) if VS_BASELINE_MODELS else []
t0 = time.time()
rows = []
for run_id, ref_id, bm, against in BM_PAIRS:
    for cell, rec in compare(Y, P[ref_id], P[run_id], CELLS, BOOT, FOLD_CODE).items():
        rows.append({**run_info(run_id), 'baseline model': bm, 'against': against, 'country': cell,
                     'baseline run': ref_id, **{k: rec.get(k, np.nan) for k in PAIR_COLS}})
cbm = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'baseline model', 'against', 'country',
                                  'baseline run'] + PAIR_COLS)
cbm = cbm.rename(columns={'R2 a': 'R2 baseline model', 'R2 b': 'R2'})
cbm = add_q(cbm, ['start', 'model', 'combo', 'baseline model', 'against'],
            where=(cbm['country'] != CONTINENT) & cbm['p'].notna())
cbm['verdict'] = verdict(cbm['dR2 low'], cbm['dR2 high'])
cbm = sort_bm(cbm)
cbm_sum = bm_summary(cbm, ['combo', 'country'])
if VS_BASELINE_MODELS:
    write_csv(cbm, os.path.join(EVAL_DIR, 'country_vs_baseline_models.csv'))
    write_csv(cbm_sum, os.path.join(EVAL_DIR, 'country_vs_baseline_models_summary.csv'))
    print(f'{len(BM_PAIRS)} comparisons with the baseline models in {len(CELLS)} cells in {time.time() - t0:.0f} s ->',
          os.path.join(EVAL_DIR, 'country_vs_baseline_models.csv'))
if not len(cbm):
    print('no comparison with the baseline models: ' + ('VS_BASELINE_MODELS = False' if not VS_BASELINE_MODELS else
          'no baseline model (BASELINE_MODELS) among the runs of 11-15 yet (run 11-15 with --array=10-11)'))

for start in [s for s in START_ORDER if len(cbm_sum) and s in set(cbm_sum['start'])]:
    for b in [b for b in BASELINE_MODELS if b in set(cbm_sum.loc[cbm_sum['start'] == start, 'baseline model'])]:
        for against in BM_AGAINST:
            t = cbm_sum[(cbm_sum['start'] == start) & (cbm_sum['baseline model'] == b) & (cbm_sum['against'] == against)]
            if not len(t):
                continue
            m = t.assign(cell=t.apply(consensus_cell, axis=1)).pivot(index='country', columns='combo', values='cell')
            m = m.reindex(index=[CONTINENT] + [c for c in COUNTRIES if c in m.index],
                          columns=sorted(m.columns, key=combo_order)).fillna('')
            m.index = [i if i == CONTINENT else country_label(i) for i in m.index]
            print(f'\n{start.upper()}: the ten models minus {bm_label(b, against)}, median ΔR² per country '
                  f'(models better / models; ▲ significantly better, ▼ worse)')
            display(m.rename(columns=short_combo))

In [ ]:
def bm_country_figure(start, b):
    t = cbm_sum[(cbm_sum['start'] == start) & (cbm_sum['baseline model'] == b) & (cbm_sum['country'] != CONTINENT)]
    refs = [a for a in BM_AGAINST if a in set(t['against'])]
    if not refs:
        return
    combos = sorted(set(t['combo']), key=combo_order)
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    norm = delta_norm(t['median dR2'], floor=0.02)
    fig, axes = plt.subplots(1, len(refs), figsize=((0.95 * len(combos) + 0.5) * len(refs) + 3.4,
                                                    0.36 * len(countries) + 2.6),
                             squeeze=False, sharey=True, constrained_layout=True)
    for ax, against in zip(axes[0], refs):
        g = t[t['against'] == against]
        med = g.pivot(index='country', columns='combo', values='median dR2').reindex(index=countries, columns=combos)
        lab = g.assign(cell=[f"{fmt_delta(d)}\n{n_b}/{n_m}" if n_m and np.isfinite(d) else '' for d, n_b, n_m in
                             zip(g['median dR2'], g['models better'], g['models'])]).pivot(
            index='country', columns='combo', values='cell').reindex(index=countries, columns=combos).fillna('')
        im = heatmap(ax, med.to_numpy(dtype=float), [country_label(c) for c in countries],
                     [short_combo(c) for c in combos], DIVERGING, norm, labels=lab.to_numpy().tolist(), fontsize=5.5)
        ax.xaxis.tick_top()
        ax.tick_params(axis='x', labelrotation=90)
        ax.set_title(f'minus {bm_label(b, against)}', fontsize=10, pad=8)
    fig.colorbar(im, ax=list(axes[0]), shrink=0.6, extend='both',
                 label=f'median ΔR² over the ten models (model − {model_label(b)})')
    fig.suptitle(f'{start.capitalize()}: the ten models against {model_label(b)} per country (text: median ΔR², models '
                 'better / models; without satclip_only and the baseline models)', x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'country_vs_baseline_models_{start}_{b}')


for start in [s for s in START_ORDER if len(cbm) and s in set(cbm['start'])]:
    for b in [b for b in BASELINE_MODELS if b in set(cbm.loc[cbm['start'] == start, 'baseline model'])]:
        bm_country_figure(start, b)
        same = cbm[(cbm['baseline model'] == b) & (cbm['against'] == 'same input')]
        combos = set(same.loc[same['start'] == start, 'combo'])
        panels = sorted(combos, key=combo_order) if DETAIL_INPUTS == 'all' else \
            [c for c in [BASELINE] + MAIN_INPUTS if c in combos] or sorted(combos, key=combo_order)
        detail_figure(same, start, 'ten_models', 'combo', 'model', f'country_vs_{b}_{start}',
                      f'{start.capitalize()}: every model of 11-15 minus {model_label(b)} with the same input, per country',
                      panels=panels,
                      cbar_label=f'ΔR² against {model_label(b)} within the country (dot: 95 % interval excludes 0)')